In [3]:
!pwd

/home/lab/rawhad/api-adapter


In [1]:
from datasets import Dataset

dataset = Dataset.from_json("data/ifbench/input_train_data.jsonl")
dataset

Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint'],
    num_rows: 95373
})

In [2]:
dataset[0]['ground_truth']

"[{'instruction_id': ['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], 'kwargs': [None, {'last_word': 'brief'}]}]"

In [3]:
dataset[0]['messages']

[{'content': 'Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be the word brief.',
  'role': 'user'}]

In [4]:
dataset[0]['constraint_type']

'multi'

In [5]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]
# Generate api completions
from anthropic import AsyncAnthropicVertex

client = AsyncAnthropicVertex(
    region=os.environ["GOOGLE_CLOUD_REGION"],
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
)

# simple completion
response = await client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=11000,
    messages=[{'role': 'user', 'content': 'say: Hello, world!'}],
    system="You are a helpful assistant.",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response.content[-1].text


'Hello, world!'

In [23]:
async def get_claude_response(messages):
    response = await client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=11000,
        messages=messages,
        thinking={"type": "enabled", "budget_tokens": 10000},
    )
    return response.content[-1].text


response = await get_claude_response(dataset[0]['messages'])
response



'Kpanlogo-is-a-percussion-instrument-while-Shamisen-is-a-string-instrument-To-summarize-Shamisen-qualifies-as-a-string-instrument-To-put-it-brief.'

In [7]:
type(dataset[0]['ground_truth'])

str

In [8]:
x = dataset[0]['ground_truth']
x

"[{'instruction_id': ['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], 'kwargs': [None, {'last_word': 'brief'}]}]"

In [9]:
eval(x)

[{'instruction_id': ['detectable_format:sentence_hyphens',
   'last_word:last_word_answer'],
  'kwargs': [None, {'last_word': 'brief'}]}]

In [10]:
_INTEGER_KWARG_FIELDS = frozenset({
    "N",
    "m",
    "n",
    "n_end",
    "n_start",
    "nth_paragraph",
    "num_bullets",
    "num_highlights",
    "num_paragraphs",
    "num_placeholders",
    "num_sections",
    "num_sentences",
    "num_words",
    "small_n",
})

def normalize_instruction_kwargs(kwargs_list):
  """Coerce integer-like instruction kwargs to ints after JSON loading."""
  normalized_kwargs = []
  for kwargs in kwargs_list:
    normalized = {}
    if kwargs is not None:
        for key, value in kwargs.items():
            if key in _INTEGER_KWARG_FIELDS and value is not None:
                normalized[key] = int(value)
            else:
                normalized[key] = value
    normalized_kwargs.append(normalized)
  return normalized_kwargs

In [11]:
import dataclasses
from typing import Dict, Optional, Union
@dataclasses.dataclass
class InputExample:
  key: int
  instruction_id_list: list[str]
  prompt: str
  kwargs: list[Dict[str, Optional[Union[str, int]]]]



In [12]:
def convert_to_input_example(example):
    ground_truth = eval(example['ground_truth'])
    return InputExample(
        key=example['key'],
        prompt=example['messages'][0]['content'],
        instruction_id_list=ground_truth[0]['instruction_id'],
        kwargs=normalize_instruction_kwargs(ground_truth[0]['kwargs'])
    )

x = convert_to_input_example(dataset[0])
x

InputExample(key='ai2-adapt-dev/tulu_v3.9_aya_100k_79773', instruction_id_list=['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], prompt='Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be the word brief.', kwargs=[{}, {'last_word': 'brief'}])

In [13]:
@dataclasses.dataclass
class OutputExample:
  instruction_id_list: list[str]
  prompt: str
  response: str
  follow_all_instructions: bool
  follow_instruction_list: list[bool]

In [14]:
import sys
sys.path.append("notebooks/ifbench/src")

In [19]:
import instructions_registry

In [20]:
def test_instruction_following_loose(
    inp,
    prompt_to_response,
):
  """Tests response for an upper bound for following instructions."""
  response = prompt_to_response[inp.prompt]
  if response is None:
      return OutputExample(
          instruction_id_list=inp.instruction_id_list,
          prompt=inp.prompt,
          response="",
          follow_all_instructions=False,
          follow_instruction_list=[False] * len(inp.instruction_id_list),
      )

  r = response.split("\n")
  response_remove_first = "\n".join(r[1:]).strip()
  response_remove_last = "\n".join(r[:-1]).strip()
  response_remove_both = "\n".join(r[1:-1]).strip()
  revised_response = response.replace("*", "")
  revised_response_remove_first = response_remove_first.replace("*", "")
  revised_response_remove_last = response_remove_last.replace("*", "")
  revised_response_remove_both = response_remove_both.replace("*", "")
  all_responses = [
      response,
      revised_response,
      response_remove_first,
      response_remove_last,
      response_remove_both,
      revised_response_remove_first,
      revised_response_remove_last,
      revised_response_remove_both,
  ]
  instruction_list = inp.instruction_id_list
  is_following_list = []

  for index, instruction_id in enumerate(instruction_list):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)

    instruction.build_description(**inp.kwargs[index])
    args = instruction.get_instruction_args()
    if args and "prompt" in args:
      instruction.build_description(prompt=inp.prompt)

    is_following = False
    for r in all_responses:
      if r.strip() and instruction.check_following(r):
        is_following = True
        break

    is_following_list.append(is_following)

  return OutputExample(
      instruction_id_list=inp.instruction_id_list,
      prompt=inp.prompt,
      response=response,
      follow_all_instructions=all(is_following_list),
      follow_instruction_list=is_following_list,
  )

In [24]:
prompt_to_response = {x.prompt: response}
r = test_instruction_following_loose(x, prompt_to_response)
r.follow_all_instructions

False

In [25]:
# lets do it for a small subset

test_subset = dataset.select(range(10))
test_subset

Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint'],
    num_rows: 10
})

In [27]:
import asyncio
from tqdm import tqdm

semaphore = asyncio.Semaphore(40)
pbar = tqdm(total=len(test_subset), desc="Processing", ncols=80)

async def get_claude_response_with_semaphore(messages):
    async with semaphore:
        try:
            ret = await get_claude_response(messages)
            return ret
        except Exception:
            return 'Error'
        finally:
            pbar.update(1)
claude_responses = await asyncio.gather(*[get_claude_response_with_semaphore(x['messages']) for x in test_subset])
pbar.close()
claude_responses[0]






Processing: 100%|███████████████████████████████| 10/10 [00:31<00:00,  3.15s/it]


'Kpanlogo-is-een-slaginstrument-Shamisen-is-een-snaarinstrument-dit-antwoord-is-brief'

In [28]:
test_subset = test_subset.add_column('claude_response', claude_responses)
test_subset






Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response'],
    num_rows: 10
})

In [ ]:
claude_rewards = []
for x in test_subset:
    inp = convert_to_input_example(x)
    prompt_to_response = {inp.prompt: x['claude_response']}
    r = test_instruction_following_loose(inp, prompt_to_response)
    claude_rewards.append(r.follow_all_instructions)

test_subset = test_subset.add_column('claude_reward', claude_rewards)
test_subset






Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward'],
    num_rows: 10
})

# train

In [36]:
!uv pip list | grep torch

torch                                    2.10.0
torch-c-dlpack-ext                       0.1.5
torchao                                  0.17.0
torchaudio                               2.10.0
torchvision                              0.25.0


In [37]:
import torch

In [38]:
import torch._dynamo
torch._dynamo.config.cache_size_limit = 256


import dotenv
dotenv.load_dotenv(override=True)
import os

In [39]:
import json
from unsloth import FastLanguageModel

import re
ptrn = re.compile(r"\\boxed\{(.*)\}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [42]:
test_subset = test_subset.map(lambda x: {
    "prompt": [
        {"role": "user", "content": (
                f"User Prompt: {x['messages'][0]['content']}\n<draft_response>{x['claude_response']}</draft_response>\n"
                "Look at the expression, and draft response and output \\boxed{CORRECT} if the final answer is correct "
                "or if its incorrect, output the corrent final answer \\boxed{n}, where n is the corrent final answer."
            )
        }
    ]
})
print(test_subset[0])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

{'key': 'ai2-adapt-dev/tulu_v3.9_aya_100k_79773', 'messages': [{'content': 'Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be the word brief.', 'role': 'user'}], 'ground_truth': "[{'instruction_id': ['detectable_format:sentence_hyphens', 'last_word:last_word_answer'], 'kwargs': [None, {'last_word': 'brief'}]}]", 'dataset': 'ifeval', 'constraint_type': 'multi', 'constraint': 'All sentences must be connected using hyphens, with no spaces between them.\tThe last word of your response should be the word brief.', 'claude_response': 'Kpanlogo-is-een-slaginstrument-Shamisen-is-een-snaarinstrument-dit-antwoord-is-brief', 'claude_reward': False, 'prompt': [{'content': 'User Prompt: Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last wo

In [50]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [52]:
DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 4096
DEFAULT_LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    gpu_memory_utilization=0.5,
)


==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [45]:
FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_laye

In [46]:

texts = [
    tokenizer.apply_chat_template(x['prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in test_subset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
)


outputs = model.fast_generate(
    texts,
    # sampling_params=sampling_params,
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")

ValueError: Unsloth: Passing text strings to `fast_generate` is only supported when `fast_inference=True` (vLLM). Since `fast_inference=False`, you must tokenize the input first:

  messages = tokenizer.apply_chat_template(
      [{"role": "user", "content": "Your prompt here"}],
      tokenize=True, add_generation_prompt=True,
      return_tensors="pt", return_dict=True
  )
  output = model.fast_generate(
      **messages.to('cuda'),
      max_new_tokens=64,
      temperature=1.0,
  )

In [ ]:

# because the model was mid-trained using peft, we don't need to add the lora layers again
model = FastLanguageModel.get_peft_model(
    model,
    r=DEFAULT_LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=DEFAULT_LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

FastLanguageModel.for_training(model)

In [ ]:
def reward_fn(prompts, completions, ground_truth, key, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    # extract the boxed answer from each response
    rewards = []
    for response, gt, k, p in zip(responses, ground_truth, key, prompts):
        gt = eval(gt)
        inp = InputExample(
            key=k,
            prompt=p,
            instruction_id_list=gt[0]['instruction_id'],
            kwargs=normalize_instruction_kwargs(gt[0]['kwargs'])
        )
        prompt_to_response = {inp.prompt: response}
        r = test_instruction_following_loose(inp, prompt_to_response)
        rewards.append(r.follow_all_instructions)
    return rewards





